# فصل 7: یادگیری ماشین کلاسیک در GeoAI

کد این نوت‌بوک عیناً از فصل 7 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۷-۲) چالش پنهان: چرا تصویر را مستقیم به مدل نمی‌دهیم؟


In [ ]:
import rasterio
import numpy as np

with rasterio.open("flood_stack.tif") as src:
    img = src.read()        # shape: (bands, H, W) e.g. (4, 1000, 1000)
    profile = src.profile

# put the bands last, then flatten to one row per pixel
bands, H, W = img.shape
table = img.transpose(1, 2, 0).reshape(-1, bands)

print(table.shape)          # (1000000, 4): a row per pixel, a column
                            # per band

## ماشین بردار پشتیبان (SVM)


In [ ]:
from sklearn.svm import SVC
model = SVC(kernel="rbf", C=1.0)   # rbf: bend the space for non-linear data
model.fit(X_train, y_train)        # learn the optimal margin
predicted = model.predict(X_new)   # classify unseen pixels
accuracy = model.score(X_test, y_test)

## جنگل تصادفی (Random Forest)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)              # 200 trees, each on a random subset
importances = model.feature_importances_ # which feature mattered most
joblib.dump(model, "rf_flood.pkl")        # save the trained model to disk

In [ ]:
predicted = model.predict(table)            # shape: (1000000,)
predicted_image = predicted.reshape(H, W)   # back to (1000, 1000)

profile.update(count=1, dtype="uint8")
with rasterio.open("predicted_map.tif", "w", **profile) as dst:
    dst.write(predicted_image.astype("uint8"), 1)

# predicted_map.tif now opens directly in ArcGIS Pro or QGIS

## خوشه‌بندی K-Means


In [ ]:
from sklearn.cluster import KMeans
model = KMeans(n_clusters=5, random_state=42)
model.fit(X_data)              # NOTE: no labels (y) here at all!
labels = model.labels_         # the cluster of each sample
centers = model.cluster_centers_   # coordinates of the 5 cluster centers

## ۷-۵) ارزیابی مدل: فراتر از یک درصد ساده


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=[
    "water",
    "forest",
    "urban",
    "soil",
]))

## ۷-۶) خطای پنهان: نشت مکانی


In [ ]:
from sklearn.model_selection import GroupKFold

# every sample of the same region (group) goes to train OR to test,
# never split across the two
gkf = GroupKFold(n_splits=5)
for train_idx, test_idx in gkf.split(X, y, groups=region_id):
    ...   # train and evaluate on spatially separated folds